# 01 — Ingest: IRS SOI + Census ACS Migration Data

Downloads two complementary state-to-state migration datasets:

1. **IRS SOI** — Tax-filer migration (income-weighted): households, people, AGI  
   Source: https://www.irs.gov/statistics/soi-tax-stats-migration-data  
   Coverage: 2011–2023 (12 year-pairs, 24 CSVs)

2. **Census ACS** — Survey-based migration (all residents, with demographics)  
   Source: https://www.census.gov/data/tables/time-series/demo/geographic-mobility/state-to-state-migration.html  
   Coverage: 2011–2024 (13 years, no 2020)

Nothing in this notebook modifies data — raw files land in `data/raw/` untouched.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import (
    load_config, ingest_irs_soi_migration, ingest_census_acs_migration,
    verify_raw_files, parse_census_acs_migration
)

cfg = load_config('config.yaml')
print(f'Project: {cfg["project_name"]}')
print(f'DuckDB:  {cfg["settings"]["duckdb_file"]}')

## 1. Download IRS SOI migration CSVs

24 files (12 years x 2 directions). Rate-limited to 1.5s between requests.  
Skips files already present in `data/raw/`.

In [ ]:
downloaded_irs = ingest_irs_soi_migration(cfg, skip_existing=True)

## 2. Download Census ACS migration Excel files

13 files (2011–2024, no 2020). Wide crosstab (2011–2023) and long format (2024+).

In [ ]:
downloaded_acs = ingest_census_acs_migration(cfg, skip_existing=True)

## 3. Verify raw files

In [ ]:
summary = verify_raw_files(cfg)
print(f'IRS CSVs: {len(summary)} files, {summary["rows"].sum():,} total rows')
summary

In [ ]:
# Census ACS files
raw_dir = Path('data/raw')
acs_files = sorted(raw_dir.glob('census_acs_migration_*'))
for f in acs_files:
    print(f'  {f.name}: {f.stat().st_size / 1024:.0f} KB')
print(f'\n{len(acs_files)} Census ACS files')

## 4. Load into DuckDB

In [ ]:
import duckdb
import pandas as pd
from datetime import date

db_path = cfg['settings']['duckdb_file']
con = duckdb.connect(db_path)

# Create _sources metadata table
con.execute("""
    CREATE TABLE IF NOT EXISTS _sources (
        table_name VARCHAR PRIMARY KEY,
        source_name VARCHAR,
        source_url VARCHAR,
        description VARCHAR,
        retrieved_date DATE,
        row_count INTEGER
    )
""")
print('_sources table ready')

In [ ]:
# Load IRS SOI CSVs
base_url = cfg['sources']['irs_soi_migration']['base_url']
today = date.today().isoformat()

for csv_file in sorted(raw_dir.glob('state*.csv')):
    table_name = csv_file.stem
    con.execute(f"DROP TABLE IF EXISTS {table_name}")
    con.execute(f"""
        CREATE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{csv_file}', header=true, all_varchar=false)
    """)
    row_count = con.execute(f"SELECT count(*) FROM {table_name}").fetchone()[0]
    
    direction = 'inflow' if 'inflow' in table_name else 'outflow'
    year_code = table_name.replace('stateinflow', '').replace('stateoutflow', '')
    year_from, year_to = f'20{year_code[:2]}', f'20{year_code[2:]}'
    
    con.execute("INSERT OR REPLACE INTO _sources VALUES (?, ?, ?, ?, ?, ?)", [
        table_name, 'IRS SOI Migration Data', f'{base_url}/{csv_file.name}',
        f'State-to-state {direction}, {year_from}-{year_to}', today, row_count,
    ])
    print(f'  {table_name}: {row_count:,} rows')

print(f'\n✓ IRS tables loaded')

In [ ]:
# Parse and load Census ACS (all years → single long-format table)
source_cfg = cfg['sources']['census_acs_migration']
all_dfs = []

for entry in source_cfg['years']:
    year = entry['year']
    ext = 'xlsx' if entry['filename'].endswith('.xlsx') else 'xls'
    filepath = raw_dir / f'census_acs_migration_{year}.{ext}'
    if not filepath.exists():
        print(f'  ⚠ missing: {filepath.name}')
        continue
    print(f'  parsing: {year}...', end=' ')
    df = parse_census_acs_migration(filepath, year)
    all_dfs.append(df)
    print(f'{len(df):,} rows')

df_census = pd.concat(all_dfs, ignore_index=True)
con.execute('DROP TABLE IF EXISTS census_acs_migration')
con.execute('CREATE TABLE census_acs_migration AS SELECT * FROM df_census')
row_count = con.execute('SELECT count(*) FROM census_acs_migration').fetchone()[0]

con.execute("INSERT OR REPLACE INTO _sources VALUES (?, ?, ?, ?, ?, ?)", [
    'census_acs_migration', 'Census Bureau ACS',
    'https://www.census.gov/data/tables/time-series/demo/geographic-mobility/state-to-state-migration.html',
    'ACS state-to-state migration flows (long format), 2011-2024 (no 2020)', today, row_count,
])
print(f'\n✓ census_acs_migration: {row_count:,} rows')

## 5. Verify

In [ ]:
con.execute("SELECT * FROM _sources ORDER BY table_name").df()

In [ ]:
# IRS: CA outflow 2022-2023 (top destinations by returns)
con.execute("""
    SELECT y2_state, y2_state_name, n1, n2, AGI
    FROM stateoutflow2223
    WHERE y1_statefips = 6
      AND y2_statefips NOT IN (96, 97, 98)
      AND y1_statefips != y2_statefips
    ORDER BY n1 DESC
    LIMIT 10
""").df()

In [ ]:
# Census ACS: CA outflow 2023 (top destinations by migrants)
con.execute("""
    SELECT origin, destination, migrants, moe
    FROM census_acs_migration
    WHERE origin = 'California' AND year = 2023
      AND destination NOT LIKE '%Total%'
      AND destination NOT LIKE '%Same%'
      AND destination NOT LIKE '%Different%'
      AND destination NOT LIKE '%Foreign%'
      AND destination NOT LIKE '%Abroad%'
      AND destination != 'California'
    ORDER BY migrants DESC
    LIMIT 10
""").df()

In [ ]:
con.close()
print('Done. DuckDB connection closed.')

---
**Next:** open `02-clean.ipynb` to standardize columns, filter summary rows, and build
unified migration flow tables across all years.